# Does ridge beat plain least squares in NLL when anchors are scarce?

The cosine sweep (`ls_vs_ridge_anchor_sweep`) showed that plain least squares collapses when the number of anchors n is close to the embedding dimension d, while ridge does not. This notebook checks the same thing in **NLL**: same real-vocabulary protocol as v5 (PubMed seed 1, 300 or 82 new words, zero-shot), but the map is fitted on only n = 0.5d, 1d, 2d random anchors (3 random subsets each) and on all anchors. Both directions get the same calibrated length. This version (v2) uses 5 random anchor subsets per n, logs the lambda chosen for every subset and the number of held-out anchors, and adds ridge with a wider lambda grid (up to 1000) to test whether lambda = 1 (the largest value of the paper grid) is too small when anchors are scarce. Output: `transplant_results_ls_vs_ridge_nll_v2/`. About 30 to 40 minutes on a T4 for the two GPT-2 pairs.


In [ ]:
%pip install -q transformers datasets scikit-learn
import os
CACHE_ROOT = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/tmp'

In [ ]:
import re, time, gc, random, pickle, traceback, logging, json
from typing import Tuple
from pathlib import Path
from collections import Counter
from types import SimpleNamespace
import numpy as np, torch, torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModel
from sklearn.linear_model import orthogonal_mp
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

In [ ]:
RUN_SET = "main"           # fixed in this file
SEED = 1                   # PubMed seed 1 (the seed of the WECHSEL runs)
DOMAIN = "pubmed"          # label used in the output file name ("pubmed" or "arxiv"); for arXiv also set DOMAIN_TXT to a file with one abstract per line
OMP64_K = 64                # the official tool's default
N_ABSTRACTS = 4000          # PubMedQA abstracts read (70% train / 30% eval, split by abstract)
TRAIN_FRAC = 0.7
K_TERMS = 300               # maximum number of new tokens added
MIN_TRAIN_COUNT, MIN_EVAL_COUNT = 5, 3     # a word must occur this often in train / eval text
MIN_TERMS = 50              # skip a pair with fewer eligible new words
MIN_PIECES = 2              # the receiver must split the word into at least this many tokens
OMP_K = 32
SALT_K = 2000              # neighbours per new token for the SALT-style local regression (our adaptation)
SALT_LAM = 1e-2            # its relative ridge strength
GRID = [0.3, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.5]    # embedding scale, relative to the mean receiver radius
N_TRAIN_T, N_TRAIN_C = 300, 150        # calibration sentences: with new words / without
N_EVAL_T, N_EVAL_C = 600, 200          # evaluation sentences: with new words / each control set
POOL_MAX = 2000                        # training pool for the optional new-row training
TRAIN_STEPS, TRAIN_LR, BATCH = 200, 1e-3, 8    # 0 = zero-shot only. Training changes ONLY the new rows
EVAL_BATCH = 16
USE_WIKI = True             # WikiText-2 as a general-text control (falls back to domain text only if it cannot load)
DOMAIN_TXT = None           # optional path: one abstract per line, to skip the PubMedQA download
OUT = Path(f"transplant_results_v5_{RUN_SET}"); OUT.mkdir(exist_ok=True)   # new folder: avoids the stale-folder skip of old results
WECHSEL_CACHE = Path(CACHE_ROOT) / "wechsel_matrices"; WECHSEL_CACHE.mkdir(parents=True, exist_ok=True)
MAIN_PAIRS = [   # tied-embedding receivers
    dict(name="GPT-2 <- Pythia-160m",     receiver="openai-community/gpt2", donor="EleutherAI/pythia-160m"),
    dict(name="GPT-2 <- Qwen2.5-0.5B",    receiver="openai-community/gpt2", donor="Qwen/Qwen2.5-0.5B"),
    dict(name="OPT-125m <- Pythia-160m",  receiver="facebook/opt-125m",     donor="EleutherAI/pythia-160m"),
    dict(name="OPT-125m <- Qwen2.5-0.5B", receiver="facebook/opt-125m",     donor="Qwen/Qwen2.5-0.5B"),
]
SCALE_PAIRS = [  # larger receivers; Pythia-1B has separate (untied) input and output matrices
    dict(name="GPT-2 large <- Qwen2.5-0.5B",  receiver="openai-community/gpt2-large", donor="Qwen/Qwen2.5-0.5B"),
    dict(name="Pythia-1B <- Qwen2.5-0.5B",    receiver="EleutherAI/pythia-1b",        donor="Qwen/Qwen2.5-0.5B"),
    dict(name="Qwen2.5-1.5B <- Pythia-410m",  receiver="Qwen/Qwen2.5-1.5B",           donor="EleutherAI/pythia-410m"),
]
PAIRS = MAIN_PAIRS if RUN_SET == "main" else SCALE_PAIRS
if RUN_SET == "scale": MIN_TERMS = 30      # larger tokenizers leave fewer words that are split; a pair with fewer than this is skipped

NS_MULT = [0.5, 1.0, 2.0]    # anchors used: n = mult x donor dimension d (plus all anchors)
DRAWS = 5                    # random anchor subsets per n
LAM_PAPER = (1e-4, 1e-3, 1e-2, 1e-1, 1.0)      # the lambda grid of the method as described in the paper
LAM_WIDE = (1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1000.0)   # extended grid, to test whether lambda = 1 (the paper grid edge) is too small when anchors are scarce
SEED = 1
OUT = Path("transplant_results_ls_vs_ridge_nll_v2"); OUT.mkdir(exist_ok=True)
SWEEP_PAIRS = MAIN_PAIRS[:2]   # GPT-2 <- Pythia-160m, GPT-2 <- Qwen2.5-0.5B


In [ ]:
def split_sents(text):
    text = " ".join(text.split())
    return [s for s in re.split(r"(?<=[.!?])\s+", text) if 8 <= len(s.split()) <= 60]

def load_domain():
    if DOMAIN_TXT:
        return [l.strip() for l in open(DOMAIN_TXT, encoding="utf-8") if l.strip()][:N_ABSTRACTS]
    from datasets import load_dataset
    for cfg, stream in (("pqa_artificial", True), ("pqa_labeled", False)):
        try:
            ds, out = load_dataset("qiaojin/PubMedQA", cfg, split="train", streaming=stream), []
            for ex in ds:
                out.append(" ".join(ex["context"]["contexts"]))
                if len(out) >= N_ABSTRACTS: break
            if out: print(f"domain text: PubMedQA/{cfg}, {len(out)} abstracts"); return out
        except Exception as e:
            print(f"PubMedQA/{cfg} failed: {type(e).__name__}: {str(e)[:150]}")
    raise RuntimeError("Could not load PubMedQA. Turn Internet on, or set DOMAIN_TXT to a text file with one abstract per line.")

def load_wiki():
    if not USE_WIKI: return []
    try:
        from datasets import load_dataset
        ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="train")
        text = " ".join(x for x in ds["text"][:30000] if x.strip() and not x.startswith(" ="))
        s = split_sents(text); print(f"general control: WikiText-2, {len(s)} sentences"); return s
    except Exception as e:
        print(f"WikiText-2 not available ({type(e).__name__}); only the domain control is used"); return []

SEG = re.compile(r" ?[^\W\d_]+| ?\d+| ?_+| ?[^\s\w]+|\s+")
def segments(s): return SEG.findall(s)

_ENC = {}
def enc(tok, seg):
    k = (id(tok), seg)
    if k not in _ENC: _ENC[k] = tok.encode(seg, add_special_tokens=False)
    return _ENC[k]

def build_seq(segs, tok, term_id, mode):
    ids, fl = [], []
    for g in segs:
        if mode == "new" and g in term_id: ids.append(term_id[g]); fl.append(1)
        else:
            p = enc(tok, g); ids += p; fl += [1 if g in term_id else 0] * len(p)
    return ids, fl

def mk(sents, tok, term_id, need_new):
    segs = [segments(s) for s in sents]
    o = [build_seq(g, tok, term_id, "orig") for g in segs]
    d = dict(sents=sents, o_ids=[x[0] for x in o], o_fl=[x[1] for x in o], n_ids=None, n_fl=None)
    if need_new:
        n = [build_seq(g, tok, term_id, "new") for g in segs]
        d["n_ids"], d["n_fl"] = [x[0] for x in n], [x[1] for x in n]
    return d

def pick_terms(rec_tok, don_tok, train_sents, eval_sents, V0r, V0d):
    ctr, cev = Counter(), Counter()
    for s in train_sents: ctr.update(g for g in segments(s)[1:] if re.fullmatch(r" [^\W\d_]{4,}", g))
    for s in eval_sents:  cev.update(g for g in segments(s)[1:] if re.fullmatch(r" [^\W\d_]{4,}", g))
    out = []
    for g, c in ctr.items():
        if c < MIN_TRAIN_COUNT or cev[g] < MIN_EVAL_COUNT: continue
        r, d = enc(rec_tok, g), enc(don_tok, g)
        if len(r) >= MIN_PIECES and len(d) == 1 and d[0] < V0d: out.append((c + cev[g], g, d[0]))
    out.sort(key=lambda x: (-x[0], x[1]))
    return out[:K_TERMS]

In [ ]:
def load_receiver(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name).float().to(DEVICE).eval()      # always float32: newer versions load the checkpoint's own precision (OPT is fp16)
    tied = model.get_output_embeddings().weight.data_ptr() == model.get_input_embeddings().weight.data_ptr()
    print(f"  receiver {name}: dtype {next(model.parameters()).dtype}, embeddings {'tied' if tied else 'UNTIED (separate output matrix)'}")
    bos = tok.bos_token_id if tok.bos_token_id is not None else tok.eos_token_id
    return tok, model, bos, tied

def load_donor(name):
    tok = AutoTokenizer.from_pretrained(name)
    m = AutoModelForCausalLM.from_pretrained(name)
    E_in = m.get_input_embeddings().weight.detach().float().cpu().numpy().copy()
    oe = m.get_output_embeddings()
    E_out = None if (oe is None or oe.weight.data_ptr() == m.get_input_embeddings().weight.data_ptr()) else oe.weight.detach().float().cpu().numpy().copy()
    del m; gc.collect(); return tok, E_in, E_out        # E_out is None when the donor ties its embeddings

def make_batch(ids_list, fl_list, idx, bos):
    L = max(len(ids_list[i]) for i in idx) + 1
    x = torch.full((len(idx), L), bos, dtype=torch.long); m = torch.zeros((len(idx), L), dtype=torch.long); f = torch.zeros((len(idx), L - 1))
    for r, i in enumerate(idx):
        s = ids_list[i]; x[r, 1:1 + len(s)] = torch.tensor(s); m[r, :1 + len(s)] = 1; f[r, :len(s)] = torch.tensor(fl_list[i], dtype=torch.float)
    return x.to(DEVICE), m.to(DEVICE), f.to(DEVICE)

def batch_nll(model, x, m, limit=None):
    logits = model(input_ids=x, attention_mask=m).logits[:, :-1].float()
    if limit is not None: logits = logits[:, :, :limit]      # the unchanged receiver: ignore the added tokens completely
    nll = F.cross_entropy(logits.reshape(-1, logits.size(-1)), x[:, 1:].reshape(-1), reduction="none").view(x.size(0), -1)
    return nll, m[:, 1:].float()

@torch.no_grad()
def score(model, bos, ids_list, fl_list, limit=None):
    n = len(ids_list); tot = np.zeros(n); term = np.zeros(n)
    order = sorted(range(n), key=lambda i: len(ids_list[i]))
    for b in range(0, n, EVAL_BATCH):
        idx = order[b:b + EVAL_BATCH]
        x, m, f = make_batch(ids_list, fl_list, idx, bos)
        nll, mm = batch_nll(model, x, m, limit)
        tot[idx] = (nll * mm).sum(1).cpu().numpy(); term[idx] = (nll * mm * f).sum(1).cpu().numpy()
    return tot, term

def set_rows(ctx, rows):
    """rows: array (K, d) for a tied receiver, or a tuple (input_rows, output_rows) for an untied one."""
    items = [(ctx.model.get_input_embeddings().weight, rows[0] if isinstance(rows, tuple) else rows)]
    if isinstance(rows, tuple): items.append((ctx.model.get_output_embeddings().weight, rows[1]))
    with torch.no_grad():
        for w, r in items: w[ctx.V0:ctx.V0 + ctx.K] = torch.as_tensor(np.asarray(r, np.float32), dtype=w.dtype, device=w.device)

def make_rows(ctx, c, d):
    if isinstance(d, tuple): return (ctx.mu_r + c * ctx.rbar * d[0], ctx.mu_ro + c * ctx.rbar_o * d[1])
    return ctx.mu_r + c * ctx.rbar * d

def unit(x, eps=1e-12): return x / np.maximum(np.linalg.norm(x, axis=-1, keepdims=True), eps)

def fit_ridge(Dc, Rc, seed=0):
    s = float(np.linalg.norm(Dc, axis=1).mean()); X = (Dc / s).astype(np.float64); Y = Rc.astype(np.float64)
    perm = np.random.default_rng(seed).permutation(len(X)); nv = min(max(50, len(X) // 10), len(X) // 5); va, tr = perm[:nv], perm[nv:]
    def solve(Xt, Yt, lam):
        A = Xt.T @ Xt; A[np.diag_indices_from(A)] += lam * np.trace(A) / A.shape[0]
        return np.linalg.solve(A, Xt.T @ Yt)
    best = None
    for lam in (1e-4, 1e-3, 1e-2, 1e-1, 1.0):
        cos = (unit(X[va] @ solve(X[tr], Y[tr], lam)) * unit(Y[va])).sum(1).mean()
        if best is None or cos > best[0]: best = (cos, lam)
    return solve(X, Y, best[1]), s, best[1]

def ridge_dirs(Dc, Rc, Dterm_c):
    W, s, lam = fit_ridge(Dc, Rc); return unit((Dterm_c / s) @ W), lam

def omp_dirs(Dc, Rc, Dterm_c, k):
    coef = orthogonal_mp(unit(Dc).astype(np.float64).T, unit(Dterm_c).astype(np.float64).T, n_nonzero_coefs=k)
    return unit(coef.reshape(len(Dc), -1).T @ unit(Rc))

def objective(ctx):
    t, c = ctx.D["t_train"], ctx.D["c_train"]
    tt, _ = score(ctx.model, ctx.bos, t["n_ids"], t["n_fl"]); cc, _ = score(ctx.model, ctx.bos, c["o_ids"], c["o_fl"])
    return 0.5 * tt.sum() / sum(map(len, t["o_ids"])) + 0.5 * cc.sum() / sum(map(len, c["o_ids"]))

def calibrate(ctx, direction):
    best = None
    for c in GRID:
        set_rows(ctx, make_rows(ctx, c, direction)); o = objective(ctx)
        if best is None or o < best[1]: best = (c, o)
    return best

def evaluate(ctx, mode):
    r = {}
    for name in ("t_eval", "c_dom", "c_wiki"):
        S = ctx.D[name]
        if S is None: continue
        use_new = mode == "new" and S["n_ids"] is not None
        ids, fl = (S["n_ids"], S["n_fl"]) if use_new else (S["o_ids"], S["o_fl"])
        tot, term = score(ctx.model, ctx.bos, ids, fl, ctx.V0 if mode == "orig" else None); r[name] = dict(tot=tot, term=term, ntok=np.array(list(map(len, ids))), nbytes=np.array([len(s.encode("utf-8")) for s in S["sents"]]))
    return r

def train_new_rows(ctx, rows, steps, seed):
    set_rows(ctx, rows)
    embs = [ctx.model.get_input_embeddings().weight]
    if not ctx.tied: embs.append(ctx.model.get_output_embeddings().weight)
    backups = [e.detach()[:ctx.V0].clone() for e in embs]
    for p in ctx.model.parameters(): p.requires_grad_(False)
    for e in embs: e.requires_grad_(True)
    opt = torch.optim.AdamW(embs, lr=TRAIN_LR, weight_decay=0.0); ctx.model.eval()
    P = ctx.D["t_pool"]; rng = random.Random(seed); skipped = 0
    for _ in range(steps):
        idx = [rng.randrange(len(P["n_ids"])) for _ in range(BATCH)]
        x, m, f = make_batch(P["n_ids"], P["n_fl"], idx, ctx.bos)
        nll, mm = batch_nll(ctx.model, x, m); loss = (nll * mm).sum() / mm.sum()
        if not torch.isfinite(loss): skipped += 1; opt.zero_grad(set_to_none=True); continue
        loss.backward()
        for e in embs: e.grad[:ctx.V0] = 0
        torch.nn.utils.clip_grad_norm_(embs, 1.0)
        if not all(bool(torch.isfinite(e.grad).all()) for e in embs): skipped += 1; opt.zero_grad(set_to_none=True); continue
        opt.step(); opt.zero_grad(set_to_none=True)
    for e in embs: e.requires_grad_(False)
    with torch.no_grad():
        for e, b in zip(embs, backups): e[:ctx.V0].copy_(b)        # the original rows are restored exactly, whatever happened
    if skipped: print(f"      note: {skipped} of {steps} training steps skipped (non-finite loss or gradient)")

# ---------- extra baselines (our implementations) ----------
def ls_dirs(Dc, Rc, Dterm_c):
    """plain least squares (lambda -> 0): minimum-norm solution of min ||X W - Y||, then direction only"""
    s = float(np.linalg.norm(Dc, axis=1).mean())
    W = np.linalg.lstsq((Dc / s).astype(np.float64), Rc.astype(np.float64), rcond=None)[0]
    return unit((Dterm_c / s) @ W)

def procrustes_dirs(Dc, Rc, Dterm_c):
    """orthogonal Procrustes between centred spaces; for different dimensions W = U V^T from the SVD of X^T Y (a partial isometry)"""
    s = float(np.linalg.norm(Dc, axis=1).mean())
    M = (Dc / s).astype(np.float64).T @ Rc.astype(np.float64)
    U, _, Vt = np.linalg.svd(M, full_matrices=False)
    return unit((Dterm_c / s) @ (U @ Vt))

def salt_pred(Dc, Rc, Dterm_c, k, lam):
    """SALT-style local regression (OUR ADAPTATION; the original code was not available): for each new token, the k nearest anchors
    in the donor space (cosine), a ridge regression WITH intercept fitted on them (dual form, k x k solve), then the prediction.
    Returns the raw prediction in centred receiver coordinates."""
    s = float(np.linalg.norm(Dc, axis=1).mean()); k = min(k, len(Dc))
    f64 = dict(dtype=torch.float64, device=DEVICE)
    Dn = torch.as_tensor(unit(Dc), **f64); Xa = torch.as_tensor(Dc / s, **f64); Ya = torch.as_tensor(Rc, **f64)
    T = torch.as_tensor(Dterm_c / s, **f64); Tn = torch.as_tensor(unit(Dterm_c), **f64); out = []
    eye = torch.eye(k, **f64)
    for t in range(len(T)):
        idx = torch.topk(Dn @ Tn[t], k).indices
        X, Y = Xa[idx], Ya[idx]; mx, my = X.mean(0), Y.mean(0); Xc, Yc = X - mx, Y - my
        G = Xc @ Xc.T; G = G + lam * torch.trace(G) / k * eye
        out.append(my + ((T[t] - mx) @ Xc.T) @ torch.linalg.solve(G, Yc))
    return torch.stack(out).cpu().numpy()

def official_omp_rows(Dtargets, Dcand, Rcand, k):
    tg = torch.as_tensor(Dtargets, dtype=torch.float32, device=DEVICE); cd = torch.as_tensor(Dcand, dtype=torch.float32, device=DEVICE)
    idx, co = batch_omp(tg, cd, k)
    rc = torch.as_tensor(Rcand, dtype=torch.float32, device=DEVICE)
    return torch.bmm(co.unsqueeze(1), rc[idx]).squeeze(1).cpu().numpy()

def build_side(Dm, Rm, r_ids, d_ids, d_term, f_idx, piece_ids, seed, wech_rows):
    """All methods for ONE embedding matrix (input side, or output side of an untied receiver).
    Dm, Rm: donor / receiver matrices of this side. Returns directions, raw rows, timing, centring constants."""
    R_a, D_a = Rm[r_ids], Dm[d_ids]; mu_r, mu_d = R_a.mean(0), D_a.mean(0); Rc, Dc = R_a - mu_r, D_a - mu_d
    rbar = float(np.linalg.norm(Rc, axis=1).mean()); Dterm_c = Dm[d_term] - mu_d; K = len(d_term)
    dirs, raw, times = {}, {}, {}
    t = time.time(); dirs["ridge[all]"], lam_all = ridge_dirs(Dc, Rc, Dterm_c); times["ridge[all]"] = time.time() - t
    t = time.time(); dirs["ridge[freq5000]"], lam_f = ridge_dirs(Dc[f_idx], Rc[f_idx], Dterm_c); times["ridge[freq5000]"] = time.time() - t
    t = time.time(); dirs["OMP-32"] = omp_dirs(Dc, Rc, Dterm_c, OMP_K); times["OMP-32 (ours, sklearn)"] = time.time() - t
    t = time.time(); omp_shipped = official_omp_rows(Dm[d_term], D_a, R_a, OMP64_K); omp_centred = official_omp_rows(Dterm_c, Dc, Rc, OMP64_K) + mu_r
    times["OMP-64 official (both versions)"] = time.time() - t
    raw["OMP-64 as shipped (raw)"] = omp_shipped; raw["OMP-64 centred (raw)"] = omp_centred
    dirs["OMP-64 centred (cal)"] = unit(omp_centred - mu_r)
    raw["mean-subword (raw)"] = np.stack([Rm[p].mean(0) for p in piece_ids]); dirs["mean-subword (cal)"] = unit(raw["mean-subword (raw)"] - mu_r)
    dirs["random (cal)"] = unit(np.random.default_rng(seed).normal(size=(K, Rm.shape[1])))
    raw["WECHSEL (raw)"] = wech_rows.astype(np.float64); dirs["WECHSEL (cal)"] = unit(wech_rows - mu_r)
    t = time.time(); dirs["LS (cal)"] = ls_dirs(Dc, Rc, Dterm_c); times["LS"] = time.time() - t
    t = time.time(); dirs["Procrustes (cal)"] = procrustes_dirs(Dc, Rc, Dterm_c); times["Procrustes"] = time.time() - t
    t = time.time(); P = salt_pred(Dc, Rc, Dterm_c, SALT_K, SALT_LAM); times["SALT-ours"] = time.time() - t
    dirs["SALT-ours (cal)"] = unit(P); raw["SALT-ours (raw)"] = mu_r + P
    return SimpleNamespace(dirs=dirs, raw=raw, times=times, mu_r=mu_r, rbar=rbar, lam=(lam_all, lam_f))


In [ ]:

def ridge_dirs_grid(Dc, Rc, Dterm_c, grid, seed=0):
    """same as the method's ridge (lambda chosen by cosine on held-out anchors: 10%, min 50, max 20%), with a configurable lambda grid; returns (directions, lambda)"""
    s = float(np.linalg.norm(Dc, axis=1).mean()); X = (Dc / s).astype(np.float64); Y = Rc.astype(np.float64)
    perm = np.random.default_rng(seed).permutation(len(X)); nv = min(max(50, len(X) // 10), len(X) // 5); va, tr = perm[:nv], perm[nv:]
    def solve(Xt, Yt, lam):
        A = Xt.T @ Xt; A[np.diag_indices_from(A)] += lam * np.trace(A) / A.shape[0]
        return np.linalg.solve(A, Xt.T @ Yt)
    best = None
    for lam in grid:
        cos = (unit(X[va] @ solve(X[tr], Y[tr], lam)) * unit(Y[va])).sum(1).mean()
        if best is None or cos > best[0]: best = (cos, lam)
    return unit((Dterm_c / s) @ solve(X, Y, best[1])), best[1], nv

def sweep_pair(pair, seed=1):
    t_all = time.time(); name = pair["name"]
    rec_tok, model, bos, tied = load_receiver(pair["receiver"]); don_tok, D_in, D_out = load_donor(pair["donor"])
    V0 = model.get_input_embeddings().weight.shape[0]; V0d = D_in.shape[0]
    E = model.get_input_embeddings().weight.detach().float().cpu().numpy().copy()
    E_out = None if tied else model.get_output_embeddings().weight.detach().float().cpu().numpy().copy()
    abstracts = load_domain(); n_tr = int(len(abstracts) * TRAIN_FRAC)
    tr_s = [s for a in abstracts[:n_tr] for s in split_sents(a)]; ev_s = [s for a in abstracts[n_tr:] for s in split_sents(a)]
    terms = pick_terms(rec_tok, don_tok, tr_s, ev_s, V0, V0d)
    print(f"{name}: {'tied' if tied else 'UNTIED'} receiver dim {E.shape[1]}, donor dim {D_in.shape[1]} | eligible new words (donor 1 token, receiver >= {MIN_PIECES} pieces): {len(terms)}")
    if len(terms) < MIN_TERMS: print(f"  fewer than {MIN_TERMS} eligible words; skipped"); return None
    for _, g, _d in terms[:12]: print(f"    {g!r:<18} receiver pieces: {[rec_tok.convert_ids_to_tokens(i) for i in enc(rec_tok, g)]}")
    K = len(terms); term_id = {g: V0 + j for j, (_, g, _d) in enumerate(terms)}
    d_term = np.array([d for _, _g, d in terms]); piece_ids = [enc(rec_tok, g) for _, g, _d in terms]
    # anchors: whole words that are single tokens in both models (matched by token string)
    rv, dv = rec_tok.get_vocab(), don_tok.get_vocab()
    keys = sorted((k for k in rv if k in dv and k.startswith("Ġ") and k[1:].isalpha() and len(k) >= 3 and rv[k] < V0 and dv[k] < V0d), key=lambda k: rv[k])
    r_ids = np.array([rv[k] for k in keys]); d_ids = np.array([dv[k] for k in keys]); print(f"  shared anchor words: {len(keys)}")
    cnt = Counter(i for s in tr_s for g in segments(s) if g not in term_id for i in enc(rec_tok, g))
    f_idx = np.argsort([-cnt[i] for i in r_ids], kind="stable")[:min(5000, len(keys))]
    # sentence sets
    rng = random.Random(seed); has = lambda s: any(g in term_id for g in segments(s))
    tr_t = [s for s in tr_s if has(s)]; tr_c = [s for s in tr_s if not has(s)]
    ev_t = [s for s in ev_s if has(s)]; ev_c = [s for s in ev_s if not has(s)]
    wiki = [s for s in load_wiki() if not has(s)]
    smp = lambda L, n: rng.sample(L, min(n, len(L)))
    c_train_s = smp(tr_c, N_TRAIN_C // 2 if wiki else N_TRAIN_C) + (smp(wiki[:len(wiki) // 2], N_TRAIN_C // 2) if wiki else [])
    D = dict(t_train=mk(smp(tr_t, N_TRAIN_T), rec_tok, term_id, True), c_train=mk(c_train_s, rec_tok, term_id, False),
             t_pool=mk(smp(tr_t, POOL_MAX), rec_tok, term_id, True), t_eval=mk(smp(ev_t, N_EVAL_T), rec_tok, term_id, True),
             c_dom=mk(smp(ev_c, N_EVAL_C), rec_tok, term_id, False),
             c_wiki=mk(smp(wiki[len(wiki) // 2:], N_EVAL_C), rec_tok, term_id, False) if wiki else None)
    print(f"  sentences: calibration {len(D['t_train']['sents'])}+{len(D['c_train']['sents'])} | pool {len(D['t_pool']['sents'])} | eval with new words {len(D['t_eval']['sents'])}, domain control {len(D['c_dom']['sents'])}, wiki control {len(D['c_wiki']['sents']) if D['c_wiki'] else 0}")
    model.resize_token_embeddings(V0 + K)
    same = model.get_output_embeddings().weight.data_ptr() == model.get_input_embeddings().weight.data_ptr()
    if same != tied: raise RuntimeError("tie status changed after resize")
    ctx = SimpleNamespace(model=model, bos=bos, V0=V0, K=K, D=D, tied=tied, mu_r=None, rbar=None, mu_ro=None, rbar_o=None)
    d_don = D_in.shape[1]; n_all = len(keys); rng = np.random.default_rng(seed)
    grid = [(f"{m:g}d", int(round(m * d_don))) for m in NS_MULT if int(round(m * d_don)) < n_all] + [("all", n_all)]
    o_ev = evaluate(ctx, "orig")["t_eval"]; o_tot = o_ev["tot"]
    out = dict(pair=name, seed=seed, K=K, d=d_don, n_all=n_all, orig=o_tot, nbytes=float(o_ev["nbytes"].mean()), runs=[])
    for tag, n in grid:
        for dr in range(1 if tag == "all" else DRAWS):
            idx = np.arange(n_all) if tag == "all" else np.sort(rng.choice(n_all, n, replace=False))
            R_a, D_a = E[r_ids[idx]], D_in[d_ids[idx]]; mu_r, mu_d = R_a.mean(0), D_a.mean(0); Rc, Dc = R_a - mu_r, D_a - mu_d
            ctx.mu_r = mu_r; ctx.rbar = float(np.linalg.norm(Rc, axis=1).mean()); Dterm_c = D_in[d_term] - mu_d
            d_p, lam_p, nv = ridge_dirs_grid(Dc, Rc, Dterm_c, LAM_PAPER); d_w, lam_w, _ = ridge_dirs_grid(Dc, Rc, Dterm_c, LAM_WIDE)
            dirs = {"LS": ls_dirs(Dc, Rc, Dterm_c), "ridge": d_p, "ridge_wide": d_w}
            rec = dict(tag=tag, n=n, draw=dr, lam=lam_p, lam_wide=lam_w, n_heldout=nv)
            for k, d in dirs.items():
                c, o = calibrate(ctx, d); set_rows(ctx, make_rows(ctx, c, d)); rec[k] = evaluate(ctx, "new")["t_eval"]["tot"]; rec[k + "_scale"] = c
            out["runs"].append(rec)
            print(f"  n = {n:>6} ({tag:>5}) draw {dr}: NLL LS {rec['LS'].mean():.2f} | ridge {rec['ridge'].mean():.2f} (lambda {lam_p:g}, {nv} held-out anchors) | ridge wide grid {rec['ridge_wide'].mean():.2f} (lambda {lam_w:g}) | unchanged receiver {o_tot.mean():.2f}")
    del ctx, model; gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    return out


In [ ]:
for pair in SWEEP_PAIRS:
    f = OUT / (pair["name"].replace(" ", "_").replace("<-", "from") + f"__seed{SEED}.pkl")
    if f.exists(): print("already done:", pair["name"]); continue
    print(f"\n########## {pair['name']} ##########")
    try:
        res = sweep_pair(pair, SEED)
        if res is not None:
            with open(f, "wb") as fh: pickle.dump(res, fh)
    except Exception as e:
        print(f"!!! failed: {type(e).__name__}: {e}"); traceback.print_exc()


In [ ]:
def boot(d, n_boot=5000):
    rng = np.random.default_rng(0); n = len(d); m = d[rng.integers(0, n, (n_boot, n))].mean(1)
    return float(d.mean()), float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5))
LN2 = float(np.log(2))
for f in sorted(OUT.glob("*.pkl")):
    r = pickle.load(open(f, "rb"))
    print(f"\n{'=' * 100}\n{r['pair']} | {r['K']} new words | d = {r['d']} | {r['n_all']} anchors | zero-shot NLL of new-word sentences (nats per sentence) | unchanged receiver {r['orig'].mean():.2f}\n{'=' * 100}")
    for name, key in (("ridge, paper grid (lambda <= 1)", "ridge"), ("ridge, wide grid (lambda <= 1000)", "ridge_wide")):
        print(f"\n{name} minus plain least squares")
        print(f"{'n':>7} {'n/d':>6} | {'LS':>8} {'ridge':>8} | per-subset differences (min .. max) | mean [CI over test sentences] | lambda chosen per subset | held-out anchors")
        for tag in dict.fromkeys(x["tag"] for x in r["runs"]):
            rs = [x for x in r["runs"] if x["tag"] == tag]; n = rs[0]["n"]
            per = np.array([(x[key] - x["LS"]).mean() for x in rs]); m, lo, hi = boot(np.mean([x[key] - x["LS"] for x in rs], 0))
            lams = [x["lam" if key == "ridge" else "lam_wide"] for x in rs]
            print(f"{n:>7} {n / r['d']:>6.2f} | {np.mean([x['LS'].mean() for x in rs]):>8.2f} {np.mean([x[key].mean() for x in rs]):>8.2f} | {per.min():+8.2f} .. {per.max():+8.2f}   | {m:+8.2f} [{lo:+8.2f}, {hi:+8.2f}] | {[f'{l:g}' for l in lams]} | {rs[0]['n_heldout']}")
